# Multi-Exchange Data Comparison

Compare crypto prices across exchanges to find arbitrage opportunities.

- Binance vs Bybit price differences
- Cross-exchange spread analysis
- Multiple timeframe analysis

In [ ]:
import json
import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1. Fetch from Multiple Exchanges

In [ ]:
def fetch_binance_klines(symbol: str, interval: str = '1h', limit: int = 100) -> pd.DataFrame:
    url = 'https://api.binance.com/api/v3/klines'
    resp = requests.get(url, params={'symbol': symbol, 'interval': interval, 'limit': limit})
    resp.raise_for_status()
    cols = ['open_time','open','high','low','close','volume',
            'close_time','quote_vol','trades','taker_buy_base','taker_buy_quote','ignore']
    df = pd.DataFrame(resp.json(), columns=cols)
    df['open_time'] = pd.to_datetime(df['open_time'], unit='ms')
    for c in ['open','high','low','close','volume']:
        df[c] = df[c].astype(float)
    df['exchange'] = 'Binance'
    return df[['open_time','open','high','low','close','volume','exchange']]


def fetch_bybit_klines(symbol: str, interval: str = '60', limit: int = 100) -> pd.DataFrame:
    """Bybit intervals: 1,3,5,15,30,60,120,240,360,720,D,W,M"""
    url = 'https://api.bybit.com/v5/market/kline'
    resp = requests.get(url, params={'category': 'spot', 'symbol': symbol,
                                      'interval': interval, 'limit': limit})
    resp.raise_for_status()
    data = resp.json()['result']['list']
    df = pd.DataFrame(data, columns=['open_time','open','high','low','close','volume','turnover'])
    df['open_time'] = pd.to_datetime(df['open_time'].astype(int), unit='ms')
    for c in ['open','high','low','close','volume']:
        df[c] = df[c].astype(float)
    df['exchange'] = 'Bybit'
    df = df[['open_time','open','high','low','close','volume','exchange']].sort_values('open_time')
    return df.reset_index(drop=True)


binance_btc = fetch_binance_klines('BTCUSDT', '1h', 100)
bybit_btc = fetch_bybit_klines('BTCUSDT', '60', 100)

print(f"Binance: {len(binance_btc)} candles, last: ${binance_btc['close'].iloc[-1]:,.2f}")
print(f"Bybit:   {len(bybit_btc)} candles, last: ${bybit_btc['close'].iloc[-1]:,.2f}")

## 2. Cross-Exchange Price Spread

In [ ]:
merged = binance_btc[['open_time', 'close']].merge(
    bybit_btc[['open_time', 'close']],
    on='open_time', suffixes=('_binance', '_bybit')
)
merged['spread_usd'] = merged['close_binance'] - merged['close_bybit']
merged['spread_bps'] = (merged['spread_usd'] / merged['close_binance']) * 10_000

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=('BTC/USDT: Binance vs Bybit', 'Price Spread (basis points)'),
    row_heights=[0.6, 0.4], vertical_spacing=0.08
)

fig.add_trace(go.Scatter(x=merged['open_time'], y=merged['close_binance'],
                          name='Binance', line=dict(color='orange')), row=1, col=1)
fig.add_trace(go.Scatter(x=merged['open_time'], y=merged['close_bybit'],
                          name='Bybit', line=dict(color='cyan')), row=1, col=1)

colors = ['green' if x > 0 else 'red' for x in merged['spread_bps']]
fig.add_trace(go.Bar(x=merged['open_time'], y=merged['spread_bps'],
                      marker_color=colors, name='Spread (bps)'), row=2, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='white', row=2, col=1)

fig.update_layout(height=600, template='plotly_dark',
                  title='Cross-Exchange Arbitrage Analysis: BTC/USDT')
fig.show()

print(f"\nSpread stats (basis points):")
print(f"  Mean: {merged['spread_bps'].mean():.2f} bps")
print(f"  Std:  {merged['spread_bps'].std():.2f} bps")
print(f"  Max:  {merged['spread_bps'].max():.2f} bps")
print(f"  Min:  {merged['spread_bps'].min():.2f} bps")

## 3. Multi-Timeframe Analysis

In [ ]:
timeframes = {'15m': '15m', '1h': '1h', '4h': '4h', '1d': '1d'}
tf_data = {}
for label, interval in timeframes.items():
    tf_data[label] = fetch_binance_klines('BTCUSDT', interval, 100)

fig = make_subplots(
    rows=2, cols=2, shared_xaxes=False,
    subplot_titles=[f'BTC/USDT {tf}' for tf in timeframes.keys()],
    vertical_spacing=0.12, horizontal_spacing=0.08
)

positions = [(1,1), (1,2), (2,1), (2,2)]
for (label, df), (r, c) in zip(tf_data.items(), positions):
    fig.add_trace(
        go.Candlestick(
            x=df['open_time'], open=df['open'], high=df['high'],
            low=df['low'], close=df['close'], name=label,
            increasing_line_color='green', decreasing_line_color='red'
        ), row=r, col=c
    )

fig.update_layout(height=800, template='plotly_dark', showlegend=False,
                  title='BTC/USDT Multi-Timeframe Analysis')
fig.update_xaxes(rangeslider_visible=False)
fig.show()

## 4. Top Movers Scanner

Scan the market for the biggest price movers in the last 24 hours.

In [ ]:
resp = requests.get('https://api.binance.com/api/v3/ticker/24hr')
tickers = pd.DataFrame(resp.json())

usdt_pairs = tickers[tickers['symbol'].str.endswith('USDT')].copy()
usdt_pairs['priceChangePercent'] = usdt_pairs['priceChangePercent'].astype(float)
usdt_pairs['quoteVolume'] = usdt_pairs['quoteVolume'].astype(float)
usdt_pairs['lastPrice'] = usdt_pairs['lastPrice'].astype(float)

# Filter for meaningful volume (> $1M daily)
active = usdt_pairs[usdt_pairs['quoteVolume'] > 1_000_000].copy()

top_gainers = active.nlargest(10, 'priceChangePercent')[['symbol', 'lastPrice', 'priceChangePercent', 'quoteVolume']]
top_losers = active.nsmallest(10, 'priceChangePercent')[['symbol', 'lastPrice', 'priceChangePercent', 'quoteVolume']]

print("=== TOP 10 GAINERS (24h) ===")
for _, r in top_gainers.iterrows():
    print(f"  {r['symbol']:>12s}  ${r['lastPrice']:>12,.4f}  {r['priceChangePercent']:>+8.2f}%  Vol: ${r['quoteVolume']:>15,.0f}")

print("\n=== TOP 10 LOSERS (24h) ===")
for _, r in top_losers.iterrows():
    print(f"  {r['symbol']:>12s}  ${r['lastPrice']:>12,.4f}  {r['priceChangePercent']:>+8.2f}%  Vol: ${r['quoteVolume']:>15,.0f}")